In [1]:
import pandas as pd
import glob

# adjust path/pattern to match your directory structure
sample_file = glob.glob("/pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE-1h/**/*.csv", recursive=True)[0]
df = pd.read_csv(sample_file, nrows=5)
print(df.columns.tolist())

['gauge_id', 'artificial_surfaces_perc', 'agricultural_areas_perc', 'forests_and_seminatural_areas_perc', 'wetlands_perc', 'water_bodies_perc']


In [2]:
df

,gauge_id,artificial_surfaces_perc,agricultural_areas_perc,forests_and_seminatural_areas_perc,wetlands_perc,water_bodies_perc
0,DE110000,7.84,44.25,47.51,0.24,0.17
1,DE110010,7.82,42.92,48.89,0.22,0.15
2,DE110020,7.21,44.28,48.20,0.11,0.20
3,DE110030,6.26,50.69,42.49,0.31,0.25
4,DE110040,3.38,57.50,39.12,0.00,0.00


In [6]:
"""
Explore the CAMELS-DE-1h dataset directory to locate the timeseries files
(as opposed to the static attribute files) and print out the available
precipitation-related columns.

Run this on bwUniCluster (or wherever the dataset path is mounted), e.g.:

    python explore_camels_de_1h.py

Adjust ROOT_DIR below if your path differs.
"""

import glob
import os

import pandas as pd

ROOT_DIR = "/pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE-1h/"


def list_top_level_structure(root_dir, max_depth=3):
    """Print directories up to max_depth so we can see how the dataset is organized."""
    print(f"\n=== Directory structure under {root_dir} (max depth {max_depth}) ===")
    root_depth = root_dir.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root_dir):
        depth = dirpath.rstrip(os.sep).count(os.sep) - root_depth
        if depth > max_depth:
            dirnames[:] = []  # don't descend further
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath) or dirpath}/")
        if depth == max_depth:
            dirnames[:] = []  # stop descending past this level


def find_candidate_files(root_dir, keywords=("timeseries", "hourly")):
    """Find CSV files whose path contains any of the given keywords."""
    all_csvs = glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True)
    candidates = [
        f for f in all_csvs if any(kw.lower() in f.lower() for kw in keywords)
    ]
    print(f"\n=== Found {len(all_csvs)} total CSV files ===")
    print(f"=== {len(candidates)} match keywords {keywords} ===")
    for f in candidates[:10]:
        print(f"  {f}")
    if len(candidates) > 10:
        print(f"  ... and {len(candidates) - 10} more")
    return candidates, all_csvs


def inspect_columns(filepath, nrows=5):
    """Read a small sample of a CSV and print its column names."""
    print(f"\n=== Columns in: {filepath} ===")
    df = pd.read_csv(filepath, nrows=nrows)
    cols = df.columns.tolist()
    for c in cols:
        print(f"  {c}")
    return cols


def filter_precip_columns(columns):
    """Return only columns that look precipitation-related."""
    return [c for c in columns if "precip" in c.lower()]


def main():
    list_top_level_structure(ROOT_DIR, max_depth=3)

    candidates, all_csvs = find_candidate_files(ROOT_DIR)

    if not candidates:
        print(
            "\nNo files matched 'timeseries'/'hourly' in the name. "
            "Falling back to inspecting the first CSV found overall — "
            "you may need to adjust the ROOT_DIR or keywords."
        )
        candidates = all_csvs[:1]

    if not candidates:
        print("No CSV files found at all under ROOT_DIR. Check the path.")
        return

    # Inspect the first candidate file in detail
    cols = inspect_columns(candidates[0])

    precip_cols = filter_precip_columns(cols)
    print("\n=== Precipitation-related columns found ===")
    if precip_cols:
        for c in precip_cols:
            print(f"  {c}")
    else:
        print("  None found in this file — check other candidate files above,")
        print("  or this may still be an attributes file rather than timeseries.")


if __name__ == "__main__":
    main()


=== Directory structure under /pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE-1h/ (max depth 3) ===
/pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE-1h//
  benchmark_models/
    HBV_simulation/
      ensemble_runs/
      HBV_CAMELS_DE_1h_benchmark_ensemble_median.zarr/
    LSTM_simulation/
      LSTM_Simulation_CAMELS_DE_1h_ensemble_median.zarr/
      ensemble_runs/
  timeseries/
    .ipynb_checkpoints/
  CAMELS_DE_1h_catchment_boundaries/
    catchments/
    gauging_stations/

=== Found 1056 total CSV files ===
=== 1040 match keywords ('timeseries', 'hourly') ===
  /pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_DEG10290.csv
  /pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_DE110560.csv
  /pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE